# 03 — Experimento RAG

Construcción, prueba y evaluación del pipeline RAG para iTimeControl.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.helpers import load_config
config = load_config('../config.yaml')
print('Config cargada')

In [ ]:
# Probar el retriever
from src.rag.retriever import Retriever

retriever = Retriever(config)

test_queries = [
    '¿Cómo registro la asistencia?',
    '¿Cómo genero un reporte?',
    '¿Cómo asigno turnos a empleados?',
]

for query in test_queries:
    results = retriever.search(query, top_k=3)
    print(f'\nQuery: {query}')
    for i, r in enumerate(results, 1):
        print(f'  [{i}] score={r["score"]:.3f} | fuente={r["source"]} | {r["text"][:80]}...')

In [ ]:
# Evaluar calidad del retriever
import numpy as np
import matplotlib.pyplot as plt

all_scores = []
for query in test_queries:
    results = retriever.search(query, top_k=5)
    scores = [r['score'] for r in results]
    all_scores.extend(scores)

print(f'Score promedio: {np.mean(all_scores):.3f}')
print(f'Score min: {np.min(all_scores):.3f}')
print(f'Score max: {np.max(all_scores):.3f}')

plt.figure(figsize=(8, 3))
plt.hist(all_scores, bins=15, color='steelblue', edgecolor='white')
plt.xlabel('Similarity Score')
plt.ylabel('Frecuencia')
plt.title('Distribución de scores de similitud del retriever')
plt.tight_layout()
plt.show()

In [ ]:
# Probar el pipeline RAG completo
from src.rag.pipeline import RAGPipeline

# Carga el modelo (puede tardar varios minutos)
rag = RAGPipeline(config)

question = '¿Cómo registro la asistencia de un empleado en iTimeControl?'
result = rag.generate(question)

print(f'Pregunta: {question}')
print(f'\nRespuesta:\n{result["answer"]}')
print(f'\nFuentes: {result["sources"]}')
print(f'Chunks usados: {result["num_chunks"]}')

In [ ]:
# Ejecutar benchmark de evaluación
from src.evaluation.benchmark import run_benchmark

results = run_benchmark(config)

if results:
    import pandas as pd
    summary = results['summary']
    df = pd.DataFrame([summary])
    print('\nMétricas de evaluación:')
    print(df.to_string(index=False))
    
    # Gráfico de métricas
    fig, ax = plt.subplots(figsize=(8, 4))
    metrics = list(summary.keys())
    values  = list(summary.values())
    bars = ax.bar(metrics, values, color=['steelblue', 'cornflowerblue', 'skyblue', 'tomato', 'seagreen'])
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.set_title('Métricas de evaluación — iTimeControl Assistant')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.show()